In [3]:
! pip install sqlalchemy pyodbc

In [4]:
# Import Libraries
import sqlalchemy
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

In [5]:
# Establish Connection
server = r"SAKS\SQLEXPRESS"
database = "HEALTHCARE_"
username = "Saks"
password = "Sql@2025"

In [6]:
# Creating SQLAlchemy engine using PyODBC
engine = sqlalchemy.create_engine(f"mssql+pyodbc://{server}/{database}?driver=SQL+Server&trusted_connection=yes") 

In [7]:
# Fetching all table names 
query = "SELECT TABLE_NAME FROM INFORMATION_SCHEMA.TABLES WHERE TABLE_TYPE = 'BASE TABLE'"
df_tables = pd.read_sql(query, engine)

print(f"All Tables are Fetched \n\n {df_tables}")

# display all the column
pd.set_option('display.max_columns', None)

All Tables are Fetched 

                             TABLE_NAME
0            Medicare_Charge_Inpatient
1           Medicare_Charge_Outpatient
2   Medicare_Provider_Charge_Inpatient
3  Medicare_Provider_Charge_Outpatient
4                 Patient_history_samp
5          Review_patient_history_samp
6               Review_transaction_coo
7                      Transaction_coo
8      DRGCodes_Global_proc_id_Mapping


In [8]:
# Fetching data from a specific table
table_name = "Medicare_Provider_Charge_Inpatient"  
Medicare_Provider_Charge_Inpatient = pd.read_sql(f"SELECT * FROM {table_name}", engine)

table_name = "Medicare_Provider_Charge_Outpatient"  
Medicare_Provider_Charge_Outpatient = pd.read_sql(f"SELECT * FROM {table_name}", engine)

table_name = "Patient_history_samp"  
Patient_history_samp = pd.read_sql(f"SELECT * FROM {table_name}", engine)

table_name = "Transaction_coo"  
Transaction_coo = pd.read_sql(f"SELECT * FROM {table_name}", engine)

table_name = "DRGCodes_Global_proc_id_Mapping"  
Global_proc_id = pd.read_sql(f"SELECT * FROM {table_name}", engine)

# Predictive Model
Building model for Inpatients to find the best hospital for their procedure

In [9]:
# Separating DRG Code and Description

Medicare_Provider_Charge_Inpatient.rename(columns={"DRG_Definition":"Procedure","Total_Discharges":"Total_Discharge","Average_Covered_Charges":"Avg_covered_charges"},inplace=True)

Medicare_Provider_Charge_Outpatient.rename(columns={"APC":"Procedure","Outpatient_Services":"Total_Discharge","Average_Estimated_Submitted_Charges":"Avg_covered_charges"},inplace=True)

In [10]:
features = ["Procedure",'Provider_Name',"Provider_State","Total_Discharge","Avg_covered_charges","Average_Total_Payments",]

In [11]:
Medicare_Provider_Charge_Inpatient[features].head(5)

,Procedure,Provider_Name,Provider_State,Total_Discharge,Avg_covered_charges,Average_Total_Payments
0,039 - EXTRACRANIAL PROCEDURES W/O CC/MCC,SOUTHEAST ALABAMA MEDICAL CENTER,AL,91,32963.078125,5777.241699
1,039 - EXTRACRANIAL PROCEDURES W/O CC/MCC,MARSHALL MEDICAL CENTER SOUTH,AL,14,15131.857422,5787.571289
2,039 - EXTRACRANIAL PROCEDURES W/O CC/MCC,ELIZA COFFEE MEMORIAL HOSPITAL,AL,24,37560.375000,5434.958496
3,039 - EXTRACRANIAL PROCEDURES W/O CC/MCC,ST VINCENT'S EAST,AL,25,13998.280273,5417.560059
4,039 - EXTRACRANIAL PROCEDURES W/O CC/MCC,SHELBY BAPTIST MEDICAL CENTER,AL,18,31633.277344,5658.333496


In [12]:
def preprocess_data(df):
    df = df[features].dropna()
    
    # Encode Categorical Columns
    le_drg = LabelEncoder()
    le_state = LabelEncoder()
    le_name = LabelEncoder()
    df['Procedure'] = le_drg.fit_transform(df['Procedure'])
    df['Provider_State'] = le_state.fit_transform(df['Provider_State'])  
    df['Provider_Name'] = le_name.fit_transform(df['Provider_Name']) 
    
    print(df.columns)
    return df, le_drg, le_state,le_name

# Preprocess Data
inpatient_df, inpatient_le_drg, inpatient_le_state,inpatient_name = preprocess_data(Medicare_Provider_Charge_Inpatient)
outpatient_df, outpatient_le_drg, outpatient_le_state,outpatient_name = preprocess_data(Medicare_Provider_Charge_Outpatient)

Index(['Procedure', 'Provider_Name', 'Provider_State', 'Total_Discharge',
       'Avg_covered_charges', 'Average_Total_Payments'],
      dtype='object')
Index(['Procedure', 'Provider_Name', 'Provider_State', 'Total_Discharge',
       'Avg_covered_charges', 'Average_Total_Payments'],
      dtype='object')


In [13]:
# randome forest
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import  mean_absolute_error, mean_squared_error, r2_score

# Train Model Function
def train_model(df):
    X = df.drop(columns=['Average_Total_Payments'])
    y = df['Average_Total_Payments']
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    
    # Evaluate Model
    y_pred = model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    print(f"Mean squared error: {mse}")
    print(f"r2_score: {r2}")
    print(f'Mean Absolute Error: {mae}')
    
    return model

In [14]:
# Train Inpatient Model
inpatient_model_reg = train_model(inpatient_df)
inpatient_model_reg

# Train Outpatient Model
outpatient_model_reg = train_model(outpatient_df)
outpatient_model_reg

Mean squared error: 6498466.809511576
r2_score: 0.8892486940699287
Mean Absolute Error: 1445.4848776145272
Mean squared error: 708.0466826826968
r2_score: 0.9896937862575733
Mean Absolute Error: 13.722704690638055


RandomForestRegressor(random_state=42)

In [15]:
# XG BOOST
import xgboost

from sklearn.metrics import  mean_absolute_error, mean_squared_error, r2_score

# Train Model Function
def train_model_XGB(df):
    X = df.drop(columns=['Average_Total_Payments'])
    y = df['Average_Total_Payments']
   
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    xgb = xgboost.XGBRegressor(objective='reg:squarederror', n_estimators=100)
    xgb.fit(X_train, y_train)
    print(xgb.feature_names_in_)
    # Evaluate Model
    y_pred = xgb.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    print(f"Mean squared error: {mse}")
    print(f"r2_score: {r2}")
    print(f'Mean Absolute Error: {mae}')
    
    return xgb

In [16]:
# Train Inpatient Model
inpatient_model = train_model_XGB(inpatient_df)
inpatient_model

# Train Outpatient Model
outpatient_model = train_model_XGB(outpatient_df)
outpatient_model

['Procedure' 'Provider_Name' 'Provider_State' 'Total_Discharge'
 'Avg_covered_charges']
Mean squared error: 4131923.838822106
r2_score: 0.9295809343084825
Mean Absolute Error: 1124.5881482703528
['Procedure' 'Provider_Name' 'Provider_State' 'Total_Discharge'
 'Avg_covered_charges']
Mean squared error: 618.0492338704959
r2_score: 0.9910037746614697
Mean Absolute Error: 13.187323023441545


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=None, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=None, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=100, n_jobs=None,
             num_parallel_tree=None, random_state=None, ...)

In [17]:
# GRADIEN BOST LINEAR REGRESSOR 

from sklearn.ensemble import GradientBoostingRegressor
# Train Model Function
def train_model_grd(df):
    X = df.drop(columns=['Average_Total_Payments'])
    y = df['Average_Total_Payments']
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    grd = GradientBoostingRegressor()
    grd.fit(X_train, y_train)
    
    # Evaluate Model
    y_pred = grd.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    print(f"Mean squared error: {mse}")
    print(f"r2_score: {r2}")
    print(f'Mean Absolute Error: {mae}')
    
    return grd

In [ ]:
# Train Inpatient Model
inpatient_model_grd = train_model_grd(inpatient_df)
inpatient_model_grd

# Train Outpatient Model
outpatient_model_grd = train_model_grd(outpatient_df)
inpatient_model_grd

Mean squared error: 11155615.756671298
r2_score: 0.8098783836678106
Mean Absolute Error: 2085.2515700774566
Mean squared error: 3809.1292444264514
r2_score: 0.944554926778493
Mean Absolute Error: 37.04803106627927


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=None, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=None, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=100, n_jobs=None,
             num_parallel_tree=None, random_state=None, ...)

In [19]:
import pandas as pd
import numpy as np

def predict_best_hospital(xgb,drg_code, state_code, le_drg, le_name ,le_state, hospital_data, top_n=5):
    # Validate DRG and State
    if drg_code not in le_drg.classes_ or state_code not in le_state.classes_:
        return f"Error: DRG Code {drg_code} or State {state_code} not found in training data."


    # Filter hospital data by state and DRG
    filtered_hospitals = hospital_data[
        (hospital_data['Provider_State'] == state_code) & 
        (hospital_data['Procedure'] == drg_code)
    ]

    if filtered_hospitals.empty:
        return f"No hospitals found for DRG Code {drg_code} in {state_code}."

    # Compute the least average charge per provider
    provider_avg_payment = (
        filtered_hospitals.groupby('Provider_Name')['Average_Total_Payments']
        .mean()
        .reset_index()
        .sort_values(by='Average_Total_Payments', ascending=True)
        .head(top_n)
    )

    return provider_avg_payment


In [38]:
# Example Usage
best_hospitals_in = predict_best_hospital(
    inpatient_model,  
    '918 - POISONING & TOXIC EFFECTS OF DRUGS W/O MCC',  
    'WI',  
    inpatient_le_drg,  
    inpatient_name,
    inpatient_le_state,  
    Medicare_Provider_Charge_Inpatient[features]
)

best_hospitals_in

,Provider_Name,Average_Total_Payments
4,"COLUMBIA ST MARY'S HOSPITAL OZAUKEE, INC",3535.384521
9,MAYO CLINIC HEALTH SYSTEM EAU CLAIRE HOSPITAL,3539.562500
6,COMMUNITY MEM HSPTL,3593.428467
19,WAUKESHA MEMORIAL HOSPITAL,3636.235352
12,MERCY MED CTR OF OSHKOSH,3640.600098


In [37]:
# Example Usage
best_hospitals_out = predict_best_hospital(
    outpatient_model,  
    '0073 - Level III Endoscopy Upper Airway',  
    'AR',  
    outpatient_le_drg, 
    outpatient_name, 
    outpatient_le_state,  
    Medicare_Provider_Charge_Outpatient[features]
)

best_hospitals_out

,Provider_Name,Average_Total_Payments
0,UAMS MEDICAL CENTER,251.396149


In [22]:
import joblib
import gzip

# Save the trained model
with gzip.open("inpatient_model.pkl.gz", "wb") as f:
    joblib.dump(inpatient_model, f, compress=3)

# Save the LabelEncoders if used
with gzip.open("le_drg.pkl.gz", "wb") as f:
    joblib.dump(inpatient_le_drg, f, compress=3)

with gzip.open("le_state.pkl.gz", "wb") as f:
    joblib.dump(inpatient_le_state, f, compress=3)

with gzip.open("le_name.pkl.gz", "wb") as f:
    joblib.dump(inpatient_name, f, compress=3)

# Train and save the outpatient model
with gzip.open("outpatient_model.pkl.gz", "wb") as f:
    joblib.dump(outpatient_model, f, compress=3)

with gzip.open("le_drg_outpatient.pkl.gz", "wb") as f:
    joblib.dump(outpatient_le_drg, f, compress=3)

with gzip.open("le_state_outpatient.pkl.gz", "wb") as f:
    joblib.dump(outpatient_le_state, f, compress=3)
    
with gzip.open("le_name_outpatient.pkl.gz", "wb") as f:
    joblib.dump(outpatient_name, f, compress=3)

print("Both models and encoders saved successfully!")

Both models and encoders saved successfully!


In [23]:
import pickle

# Assuming hospital_data is your DataFrame
hospital_data_out = Medicare_Provider_Charge_Outpatient[features]

# Store the hospital data as a pickle file
with open('hospital_data_out.pkl', 'wb') as f:
    pickle.dump(hospital_data_out, f)

hospital_data_in = Medicare_Provider_Charge_Inpatient[features]

# Store the hospital data as a pickle file
with open('hospital_data_in.pkl', 'wb') as f:
    pickle.dump(hospital_data_in, f)

print("Hospital data saved to pickle file.")


Hospital data saved to pickle file.


In [32]:
def load_model(filename):
    with gzip.open(filename, "rb") as f:
        return joblib.load(f)

def load_pickle(filename):
    with open(filename, "rb") as f:
        return pickle.load(f)

# Load models and encoders
inpatient_model = load_model("inpatient_model.pkl.gz")
outpatient_model = load_model("outpatient_model.pkl.gz")
inpatient_le_drg = load_model("le_drg.pkl.gz")
inpatient_le_state = load_model("le_state.pkl.gz")
inpatient_le_name = load_model("le_name.pkl.gz")
outpatient_le_drg = load_model("le_drg_outpatient.pkl.gz")
outpatient_le_state = load_model("le_state_outpatient.pkl.gz")
outpatient_le_name = load_model("le_name_outpatient.pkl.gz")
hospital_data_in = load_pickle("hospital_data_in.pkl")
hospital_data_out = load_pickle("hospital_data_out.pkl")

In [31]:
hospital_data_out.columns

Index(['Procedure', 'Provider_Name', 'Provider_State', 'Total_Discharge',
       'Avg_covered_charges', 'Average_Total_Payments'],
      dtype='object')